In [0]:
from bggAPI.BGGApi import BGGThing
from bggAPI.BGGDataCleaner import BGGDataCleaner

In [0]:
%sql
TRUNCATE TABLE boardgame.boardgamegeek.src__bgg__thing;


In [0]:
game_data = []
version_data_final = []
version_link_data = []
version_publisher_data = []
version_language_data = []
attribute_data = {}
attribute_link_data = {}
alt_name_data = []
poll_data_final = {}
mp_data_final = []

thing_api = BGGThing()


ids = [5, 
       13,  
       552, 
       2651,
       12692,
       97207,
       102794,
       105551, 
       125153, 
       216132, 
       233247, 
       311193, 
       329591, 
       347305, 
       388367]

for id in ids:
    data = thing_api.get_data(id)
    
    if data is not None:
        thing_clean = BGGDataCleaner(data)
        
        clean_data, att, alt_name, version_data, poll_data, mp_data = thing_clean.multiple_clean()        
        
        game_data = game_data + clean_data
        

        if len(version_data)>0:
            version_data_final = version_data_final + version_data["version"]
            version_link_data = version_link_data + version_data["bg_version"]
            version_publisher_data = version_publisher_data + version_data["bg_publisher"]
            version_language_data = version_language_data + version_data["language"]
        
        for att_name in att["attribute"].keys():

            if att_name not in attribute_data:
                attribute_data[att_name] = []
                attribute_link_data[att_name] = []

            attribute_data[att_name] = attribute_data[att_name] + att["attribute"][att_name]
            attribute_link_data[att_name] = attribute_link_data[att_name] + att["link"][att_name] 
            
    
        if len(alt_name)>0:
            alt_name_data = alt_name_data + alt_name

        if len(poll_data)>0:
            for poll in poll_data.keys():
                if poll not in poll_data_final:
                    poll_data_final[poll] = []
                poll_data_final[poll] = poll_data_final[poll] + poll_data[poll]
        
        if len(mp_data)>0:
            mp_data_final = mp_data_final + mp_data


In [0]:

if len(game_data) > 0:
  game_sp_df = spark.createDataFrame(game_data)

  version_sp_df = spark.createDataFrame(version_data_final)
  version_link_sp_df = spark.createDataFrame(version_link_data)
  version_publisher_sp_df = spark.createDataFrame(version_publisher_data)
  version_language_sp_df = spark.createDataFrame(version_language_data)

  game_sp_df.write.format("delta").mode("append").saveAsTable("boardgame.boardgamegeek.src__bgg__thing")

  version_sp_df.write.format("delta").mode("append").saveAsTable("boardgame.boardgamegeek.src__bgg__versions")
  version_link_sp_df.write.format("delta").mode("append").saveAsTable("boardgame.boardgamegeek.src__bgg__version_links")
  version_publisher_sp_df.write.format("delta").mode("append").saveAsTable("boardgame.boardgamegeek.src__bgg__version_publisher_links")
  version_language_sp_df.write.format("delta").mode("append").saveAsTable("boardgame.boardgamegeek.src__bgg__version_language_links")

  for att_name in attribute_data.keys():
    df_sp_att = spark.createDataFrame(attribute_data[att_name])
    df_sp_att = df_sp_att.dropDuplicates()
    df_sp_att.write.format("delta").mode("append").saveAsTable(f"boardgame.boardgamegeek.src__bgg__boardgame_{att_name}")


    df_sp_att_l = spark.createDataFrame(attribute_link_data[att_name])
    df_sp_att_l = df_sp_att_l.dropDuplicates()
    df_sp_att_l.write.format("delta").mode("append").saveAsTable(f"boardgame.boardgamegeek.src__bgg__boardgame_{att_name}_links")

  df_sp_alt_name = spark.createDataFrame(alt_name_data)
  df_sp_alt_name.write.format("delta").mode("append").saveAsTable("boardgame.boardgamegeek.src__bgg__boardgame_alt_name")

  for poll in poll_data_final.keys():
      df_poll = spark.createDataFrame(poll_data_final[poll])
      df_poll.write.format("delta").mode("append").saveAsTable(f"boardgame.boardgamegeek.src__bgg__poll_{poll}")

if len(mp_data_final) >0:
  df_mp = spark.createDataFrame(mp_data_final)
  df_mp.write.format("delta").mode("append").saveAsTable("boardgame.boardgamegeek.src__bgg__marketplace_listings")